# Khởi tạo và đóng băng mô hình VGG16 (Model initialization and freezing)
Mục tiêu:
- Tải kiến trúc mạng VGG16 cùng bộ trọng số đã được huấn luyện sẵn từ tập dữ liệu ImageNet.
- Thực hiện kỹ thuật đóng băng (Freeze) toàn bộ phần thân (lớp Convolutional) của mô hình nhằm giữ lại các đặc trưng hình ảnh tổng quát và tiết kiệm tài nguyên tính toán của máy Mac.
- Kiểm tra trạng thái của các tham số để đảm bảo quá trình đóng băng diễn ra chính xác trước khi sang bước tiếp theo.

In [1]:
import torch
import torch.nn as nn
from torchvision import models

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Sử dụng thiết bị CUDA: {device}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Sử dụng thiết bị Apple Metal Performance Shaders: {device}")
else:
    device = torch.device("cpu")
    print(f"Sử dụng CPU: {device}")

Sử dụng thiết bị Apple Metal Performance Shaders: mps


## Tải mô hình VGG16 Pretrained
Sử dụng bộ trọng số mặc định `VGG16_Weights.DEFAULT` để nạp các tri thức trích xuất đặc trưng đã được tối ưu hóa từ kho ảnh ImageNet khổng lồ.

In [2]:
base_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
base_model = base_model.to(device)

print(f"Mô hình đã được nạp và chuyển lên thiết bị thành công.")

Mô hình đã được nạp và chuyển lên thiết bị thành công.


# Đóng băng phần thân
Duyệt qua toàn bộ tham số của khối trích xuất đặc trưng (`base_model.features`) và tắt bộ đếm gradient bằng thuộc tính `requires_grad = False`.

In [3]:
# Thực hiện đóng băng các lớp Tích chập
for param in base_model.features.parameters():
    param.requires_grad = False

print("Đã đóng băng toàn bộ phần thân (Features) của VGG16.")

Đã đóng băng toàn bộ phần thân (Features) của VGG16.


## Xác minh trạng thái Gradient đa nền tảng
In kiểm tra trạng thái khóa học tập của một số lớp đại diện để chắc chắn mô hình sẵn sàng cho bước tiếp theo mà không gây lãng phí tài nguyên tính toán.

In [4]:
print("Kiểm tra trạng thái Gradient phần thân")
for i, (name, param) in enumerate(base_model.features.named_parameters()):
    if i < 3:
        print(f"Lớp: {name} | Được huấn luyện (Trainable): {param.requires_grad}")

print("\nKiểm tra trạng thái Gradient phần nguyên bản")
for i, (name, param) in enumerate(base_model.classifier.named_parameters()):
    if i < 3:
        print(f"Lớp: {name} | Được huấn luyện (Trainable): {param.requires_grad}")

Kiểm tra trạng thái Gradient phần thân
Lớp: 0.weight | Được huấn luyện (Trainable): False
Lớp: 0.bias | Được huấn luyện (Trainable): False
Lớp: 2.weight | Được huấn luyện (Trainable): False

Kiểm tra trạng thái Gradient phần nguyên bản
Lớp: 0.weight | Được huấn luyện (Trainable): True
Lớp: 0.bias | Được huấn luyện (Trainable): True
Lớp: 3.weight | Được huấn luyện (Trainable): True
